<h1 style="color:#ffffff; background-color:#0B4F45; text-align:center; font-weight:bold; padding:16px 10px; border-radius:8px; margin-bottom:6px;">Day 2 — Cross-Validation</h1>
<h3 style="color:#0F5C52; text-align:center; font-weight:bold; margin-top:0;">Trusting a Score Means Not Trusting Just One Split</h3>


<a id="toc"></a>
<h2 style="color:#7FE8D3; background-color:#0F5C52; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">Table of Contents</h2>
<div style="border:1px solid #ccc; padding:15px 25px; border-radius:6px;">
<ol style="font-weight:bold; line-height:1.9;">
<li><a href="#section0">0. Setup — Importing Pandas, NumPy &amp; Scikit-learn</a></li>
<li><a href="#section1">1. What Cross-Validation Does</a></li>
<li><a href="#section2">2. How k-Fold Works (k = 5)</a></li>
<li><a href="#section3">3. cross_val_score — Mean &amp; Standard Deviation</a></li>
<li><a href="#section4">4. Stratified k-Fold for Classification</a></li>
<li><a href="#section5">5. Common Mistakes to Avoid</a></li>
<li><a href="#section6">6. Quick Reference</a></li>
<li><a href="#section7">7. Hands-On Lab — Cross-Validating a Model</a></li>
<li><a href="#section8">8. Best Practices &amp; Reproducibility</a></li>
<li><a href="#section9">9. Summary — What I Learned Today</a></li>
</ol>
</div>


<a id="section0"></a>
<h2 style="color:#7FE8D3; background-color:#0F5C52; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">0. Setup — Importing Pandas, NumPy &amp; Scikit-learn</h2>
<div style="border-left:5px solid #4A9DE0; background-color:#eaf4fc; padding:10px 15px; margin:10px 0; border-radius:4px; color:#1c4e6e;">
<b>Note:</b> Yesterday's three-way split gave us one honest validation score and one honest test score. But a <b>single</b> validation split is still just one roll of the dice. Today we replace that one roll with several, and average them — that is what <b>cross-validation</b> means.
</div>


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

print("Pandas version      :", pd.__version__)
print("NumPy version       :", np.__version__)
print("Scikit-learn version:", sklearn.__version__)

Pandas version      : 2.3.3
NumPy version       : 2.3.5
Scikit-learn version: 1.7.2


<a id="section1"></a>
<h2 style="color:#7FE8D3; background-color:#0F5C52; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">1. What Cross-Validation Does</h2>
<h3 style="color:#7FE8D3; font-weight:bold; background-color:#123B34; display:inline-block; padding:4px 10px; border-radius:6px;">1.1 One split can be lucky — or unlucky</h3>
In Day 1 we tuned <code>max_depth</code> using a <b>single</b> validation set. But what if that particular 20% slice happened to be easier (or harder) than the rest of the data? The score we trusted would be partly luck, not signal.

<h3 style="color:#7FE8D3; font-weight:bold; background-color:#123B34; display:inline-block; padding:4px 10px; border-radius:6px;">1.2 The fix: k-fold cross-validation</h3>
<b>k-fold cross-validation</b> replaces one lucky-or-unlucky validation split with <b>k</b> of them. It splits the training data into k equal parts ("folds"), then trains k times — each time using a <i>different</i> fold as the validation set and the other k−1 folds for training. Averaging the k scores gives a far more stable, trustworthy estimate of performance than any single split.

In [13]:
# A small demonstration: the SAME model, scored on 5 different random validation slices
# of the same data, can look meaningfully different just from the luck of the split.
np.random.seed(1)
validation_scores_across_splits = np.random.normal(loc=0.80, scale=0.04, size=5)

for i, score in enumerate(validation_scores_across_splits, start=1):
    print(f"Random validation split {i}: score = {score:.3f}")

print()
print(f"Range across splits: {validation_scores_across_splits.min():.3f} to "
      f"{validation_scores_across_splits.max():.3f}")
print("Averaging several splits instead of trusting one is exactly what k-fold does.")

Random validation split 1: score = 0.865
Random validation split 2: score = 0.776
Random validation split 3: score = 0.779
Random validation split 4: score = 0.757
Random validation split 5: score = 0.835

Range across splits: 0.757 to 0.865
Averaging several splits instead of trusting one is exactly what k-fold does.


<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>Important:</b> A high mean with a <b>low</b> standard deviation across folds is a model you can trust; a high mean with a <b>high</b> standard deviation may just mean the model got lucky on some folds.
</div>


<a id="section2"></a>
<h2 style="color:#7FE8D3; background-color:#0F5C52; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">2. How k-Fold Works (k = 5)</h2>
<h3 style="color:#7FE8D3; font-weight:bold; background-color:#123B34; display:inline-block; padding:4px 10px; border-radius:6px;">2.1 Rotating which fold validates</h3>
The training data is split into 5 equal folds. Across 5 rounds, every fold gets exactly one turn as the validation set, while the other 4 folds are used for training:

<div style="border:1px solid #ccc; border-radius:6px; overflow:hidden;">
<table style="width:100%; border-collapse:collapse;">
<tr style="background-color:#0F5C52; color:#7FE8D3;"><th style="padding:8px; text-align:left;">Round</th><th style="padding:8px; text-align:left;">Trains On</th><th style="padding:8px; text-align:left;">Validates On</th></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">1</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Folds 2, 3, 4, 5</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Fold 1</td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">2</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Folds 1, 3, 4, 5</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Fold 2</td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">3</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Folds 1, 2, 4, 5</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Fold 3</td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">4</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Folds 1, 2, 3, 5</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Fold 4</td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">5</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Folds 1, 2, 3, 4</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Fold 5</td></tr>
</table>
</div>


<div style="border-left:5px solid #4A9DE0; background-color:#eaf4fc; padding:10px 15px; margin:10px 0; border-radius:4px; color:#1c4e6e;">
<b>Note:</b> Every data point is used for validation <b>exactly once</b> and for training <b>k−1 times</b>, so no data is wasted and no single split can dominate the result. <code>k=5</code> or <code>k=10</code> are the common choices.
</div>


<a id="section3"></a>
<h2 style="color:#7FE8D3; background-color:#0F5C52; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">3. cross_val_score — Mean &amp; Standard Deviation</h2>
<h3 style="color:#7FE8D3; font-weight:bold; background-color:#123B34; display:inline-block; padding:4px 10px; border-radius:6px;">3.1 One function, k trained models</h3>
Scikit-learn automates the whole rotation with <code>cross_val_score</code>: give it a model, the data, and <code>cv</code> (the number of folds), and it returns one score per fold.

In [14]:
from sklearn.model_selection import cross_val_score

# Illustrative only -- we run this for real on the Titanic dataset in Section 7
# scores = cross_val_score(model, X_train, y_train, cv=5, scoring="f1")
# print(scores)          # one score per fold
# print(scores.mean())   # the reliable average
# print(scores.std())    # how much the score varies across folds

print("We will run cross_val_score for real on the Titanic dataset in Section 7.")

We will run cross_val_score for real on the Titanic dataset in Section 7.


<div style="border-left:5px solid #E85D9A; background-color:#fdeef4; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a2b52;">
<b>Tip:</b> <code>scores.mean()</code> is your performance estimate; <code>scores.std()</code> tells you how <b>stable</b> it is across folds — read them together, never the mean alone.
</div>


<a id="section4"></a>
<h2 style="color:#7FE8D3; background-color:#0F5C52; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">4. Stratified k-Fold for Classification</h2>
<h3 style="color:#7FE8D3; font-weight:bold; background-color:#123B34; display:inline-block; padding:4px 10px; border-radius:6px;">4.1 Why plain k-fold can mislead on classification</h3>
On classification problems — especially <b>imbalanced</b> ones — plain k-fold can accidentally create folds with very different class proportions (e.g. one fold with mostly survivors, another with mostly non-survivors). That distorts every score computed from it.

<h3 style="color:#7FE8D3; font-weight:bold; background-color:#123B34; display:inline-block; padding:4px 10px; border-radius:6px;">4.2 The fix: preserving class balance in every fold</h3>
<b>Stratified k-fold</b> fixes this by preserving the original class balance in <i>every</i> fold. Scikit-learn applies it <b>automatically</b> whenever you cross-validate a classifier with <code>cross_val_score</code> or <code>GridSearchCV</code> — this directly addresses the class-imbalance problem raised back in Week 3.

In [15]:
# Demonstrating the class-imbalance risk that stratification protects against
from sklearn.model_selection import KFold, StratifiedKFold

toy_labels = np.array([0]*80 + [1]*20)  # an 80/20 imbalanced toy target

print("Plain KFold -- class balance can drift per fold:")
for i, (_, val_idx) in enumerate(KFold(n_splits=5, shuffle=False).split(toy_labels), start=1):
    fold_labels = toy_labels[val_idx]
    print(f"  Fold {i}: {fold_labels.mean():.0%} class 1")

print()
print("StratifiedKFold -- class balance is preserved every time:")
for i, (_, val_idx) in enumerate(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(toy_labels, toy_labels), start=1):
    fold_labels = toy_labels[val_idx]
    print(f"  Fold {i}: {fold_labels.mean():.0%} class 1")

Plain KFold -- class balance can drift per fold:
  Fold 1: 0% class 1
  Fold 2: 0% class 1
  Fold 3: 0% class 1
  Fold 4: 0% class 1
  Fold 5: 100% class 1

StratifiedKFold -- class balance is preserved every time:
  Fold 1: 20% class 1
  Fold 2: 20% class 1
  Fold 3: 20% class 1
  Fold 4: 20% class 1
  Fold 5: 20% class 1


<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>Important:</b> For a classifier, always confirm you are getting <b>stratified</b> folds when it matters — Scikit-learn does this by default for classification tasks, but it is worth checking explicitly on imbalanced data.
</div>


<a id="section5"></a>
<h2 style="color:#7FE8D3; background-color:#0F5C52; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">5. Common Mistakes to Avoid</h2>
<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>Important:</b> <ul>
<li><b>Cross-validating on data that includes the test set</b> — cross-validation must run on <code>X_train</code>/<code>y_train</code> only; the test set stays untouched, exactly as in Day 1.</li>
<li><b>Reading only the mean, never the standard deviation</b> — a high mean with a huge spread across folds is not a model you can trust yet.</li>
<li><b>Ignoring stratification on imbalanced classification data</b> — unstratified folds can silently distort every score.</li>
<li><b>Forgetting <code>random_state</code></b> — without it, re-running the notebook reshuffles the folds and the reported scores drift.</li>
<li><b>Treating a single cross-validated score as the final answer</b> — it replaces the single validation split, not the one-time, untouched test set.</li>
</ul>
</div>


<a id="section6"></a>
<h2 style="color:#7FE8D3; background-color:#0F5C52; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">6. Quick Reference</h2>


<div style="border:1px solid #ccc; border-radius:6px; overflow:hidden;">
<table style="width:100%; border-collapse:collapse;">
<tr style="background-color:#0F5C52; color:#7FE8D3;"><th style="padding:8px; text-align:left;">Task</th><th style="padding:8px; text-align:left;">Code</th></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Import the tool</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>from sklearn.model_selection import cross_val_score</code></td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Run 5-fold cross-validation</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>scores = cross_val_score(model, X_train, y_train, cv=5, scoring="f1")</code></td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Reliable average</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>scores.mean()</code></td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Stability across folds</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>scores.std()</code></td></tr>
<tr style="background-color:#ffffff;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Force stratified folds explicitly</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;"><code>from sklearn.model_selection import StratifiedKFold</code></td></tr>
<tr style="background-color:#f7f7f9;"><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Golden rule</td><td style="padding:8px; border-top:1px solid #eee; color:#1a1a1a;">Cross-validate on the training set only — the test set stays untouched</td></tr>
</table>
</div>


<a id="section7"></a>
<h2 style="color:#7FE8D3; background-color:#0F5C52; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">7. Hands-On Lab — Cross-Validating a Model</h2>
<div style="border-left:5px solid #9B6FD4; background-color:#f3edfb; padding:10px 15px; margin:10px 0; border-radius:4px; color:#4d3475;">
<b>Goal:</b> Take the Day 1 Titanic model, evaluate it with 5-fold cross-validation, report the mean and standard deviation, compare it to the Day 1 single-split score, and confirm stratified folds are being used.
</div>


<h3 style="color:#7FE8D3; font-weight:bold; background-color:#123B34; display:inline-block; padding:4px 10px; border-radius:6px;">7.0 Loading the dataset</h3>
We reuse <code>train_and_test2.csv</code>, the same Titanic passenger dataset from Day 1, predicting whether a passenger survived.

In [16]:
titanic = pd.read_csv("train_and_test2.csv")
titanic = titanic.rename(columns={"2urvived": "Survived"})

features = ["Age", "Fare", "Sex", "sibsp", "Parch", "Pclass"]
X = titanic[features]
y = titanic["Survived"]

print("Shape:", X.shape)
titanic.head()

Shape: (1309, 6)


,Passengerid,Age,Fare,Sex,sibsp,zero,zero.1,zero.2,zero.3,zero.4,...,zero.12,zero.13,zero.14,Pclass,zero.15,zero.16,Embarked,zero.17,zero.18,Survived
0,1,22.0,7.2500,0,1,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0
1,2,38.0,71.2833,1,1,0,0,0,0,0,...,0,0,0,1,0,0,0.0,0,0,1
2,3,26.0,7.9250,1,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,1
3,4,35.0,53.1000,1,1,0,0,0,0,0,...,0,0,0,1,0,0,2.0,0,0,1
4,5,35.0,8.0500,0,0,0,0,0,0,0,...,0,0,0,3,0,0,2.0,0,0,0


In [17]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

<h3 style="color:#7FE8D3; font-weight:bold; background-color:#123B34; display:inline-block; padding:4px 10px; border-radius:6px;">Step 1 — Take the Day 1 model and evaluate it with 5-fold cross-validation using cross_val_score</h3>


In [22]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier

# Same three-way split discipline as Day 1: carve off the test set first, and never touch it here
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42)

# The Day 1 model: RandomForest with max_depth chosen via the validation set
model = RandomForestClassifier(max_depth=5, n_estimators=100, random_state=42)

# Cross-validate on the TRAINING set only -- the held-out test set is never touched here
cv_scores = cross_val_score(
    model,
    X_train,
    y_train,
    cv=skf,
    scoring="f1"
)

print("Fold scores:", np.round(cv_scores, 3))

Fold scores: [0.441 0.597 0.5   0.464 0.522]


<h3 style="color:#7FE8D3; font-weight:bold; background-color:#123B34; display:inline-block; padding:4px 10px; border-radius:6px;">Step 2 — Report the mean and standard deviation of the scores across folds</h3>


In [19]:
print(f"Mean F1 across 5 folds : {cv_scores.mean():.3f}")
print(f"Std. dev. across folds : {cv_scores.std():.3f}")

Mean F1 across 5 folds : 0.505
Std. dev. across folds : 0.054


<div style="border-left:5px solid #4A9DE0; background-color:#eaf4fc; padding:10px 15px; margin:10px 0; border-radius:4px; color:#1c4e6e;">
<b>Note:</b> A small standard deviation next to a solid mean means the score is <b>stable</b> — the model performs similarly no matter which slice of the training data becomes the validation fold.<br><br>
In this experiment, the model achieved a mean F1-score of 0.505 with
a standard deviation of 0.054. This indicates some variation across
folds, but the results are reasonably consistent overall.
</div>


<h3 style="color:#7FE8D3; font-weight:bold; background-color:#123B34; display:inline-block; padding:4px 10px; border-radius:6px;">Step 3 — Compare the cross-validated estimate to the single-split score from Day 1 and explain any difference</h3>


In [20]:
# The Day 1-style single validation score, for comparison
model.fit(X_train, y_train)
from sklearn.metrics import f1_score
single_split_f1 = f1_score(y_val, model.predict(X_val))

print(f"Day 1 single validation-split F1 : {single_split_f1:.3f}")
print(f"Day 2 5-fold cross-validated mean : {cv_scores.mean():.3f}  (± {cv_scores.std():.3f})")

print()
print("The single-split score is one point estimate -- it could sit anywhere within the spread")
print("shown by the cross-validated fold scores above. The cross-validated mean is the more")
print("trustworthy number because it is not dependent on which 25% happened to become X_val.")

Day 1 single validation-split F1 : 0.571
Day 2 5-fold cross-validated mean : 0.505  (± 0.054)

The single-split score is one point estimate -- it could sit anywhere within the spread
shown by the cross-validated fold scores above. The cross-validated mean is the more
trustworthy number because it is not dependent on which 25% happened to become X_val.


<div style="border-left:5px solid #E85D9A; background-color:#fdeef4; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a2b52;">
<b>Tip:</b> If the single-split score and the cross-validated mean are close, that is reassuring — but it is the <b>standard deviation</b>, not the closeness of one lucky split, that actually tells you whether to trust the number.
</div>


<h3 style="color:#7FE8D3; font-weight:bold; background-color:#123B34; display:inline-block; padding:4px 10px; border-radius:6px;">Step 4 — For a classification task, confirm stratified folds are used and explain why that matters here</h3>


In [21]:
from sklearn.model_selection import StratifiedKFold

# cross_val_score already uses stratified folds by default for a classifier -- confirming explicitly:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
stratified_scores = cross_val_score(model, X_train, y_train, cv=skf, scoring="f1")

print("Overall survival rate in y_train:", f"{y_train.mean():.1%}")
for i, (_, val_idx) in enumerate(skf.split(X_train, y_train), start=1):
    fold_rate = y_train.iloc[val_idx].mean()
    print(f"  Fold {i} survival rate: {fold_rate:.1%}")

print()
print(f"Explicit StratifiedKFold mean F1: {stratified_scores.mean():.3f}")

Overall survival rate in y_train: 26.0%
  Fold 1 survival rate: 25.5%
  Fold 2 survival rate: 26.1%
  Fold 3 survival rate: 26.1%
  Fold 4 survival rate: 26.1%
  Fold 5 survival rate: 26.1%

Explicit StratifiedKFold mean F1: 0.505


<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>Why Stratified Folds Matter</b><br>
The survival rate in the training data is about 26%, and each validation
fold keeps approximately the same class proportion.<br>
This is important because it prevents some folds from containing too many
or too few survivors, making the F1-scores across folds more representative
and comparable.
</div>


<a id="section8"></a>
<h2 style="color:#7FE8D3; background-color:#0F5C52; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">8. Best Practices &amp; Reproducibility</h2>
<div style="border-left:5px solid #F0964B; background-color:#fef2e7; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a4a13;">
<b>Important:</b> <ul>
<li>Always cross-validate on <code>X_train</code>/<code>y_train</code> only — the test set from Day 1 stays sealed until the Day 5 final evaluation.</li>
<li>Report the mean <b>and</b> the standard deviation together — never the mean alone.</li>
<li>For classification, confirm folds are <b>stratified</b>, especially on imbalanced targets like this one.</li>
<li>Fix <code>random_state=42</code> on every split and every <code>StratifiedKFold</code>, for a fully reproducible cross-validation run.</li>
<li><code>k=5</code> or <code>k=10</code> are reasonable defaults — higher k means more folds trained, at higher compute cost, for a slightly more stable estimate.</li>
</ul>
</div>


<a id="section9"></a>
<h2 style="color:#7FE8D3; background-color:#0F5C52; font-weight:bold; margin-top:30px; padding:8px 14px; border-radius:6px;">9. Summary — What I Learned Today</h2>
<div style="border-left:5px solid #E85D9A; background-color:#fdeef4; padding:10px 15px; margin:10px 0; border-radius:4px; color:#8a2b52;">
<b>Tip:</b> <ul>
<li>A <b>single</b> validation split is one roll of the dice — <b>k-fold cross-validation</b> replaces it with k rolls and averages them for a more stable estimate.</li>
<li>k-fold rotates which fold validates across k rounds, so every data point is used for validation exactly once and for training k−1 times.</li>
<li><code>cross_val_score</code> returns one score per fold; read the <b>mean</b> (the estimate) together with the <b>standard deviation</b> (how stable it is).</li>
<li><b>Stratified k-fold</b> preserves class balance in every fold, and Scikit-learn applies it automatically for classifiers — important for imbalanced data like Titanic survival.</li>
<li>Cross-validation always runs on the training set only; the untouched test set from Day 1 remains reserved for the one-time final check.</li>
</ul>
</div>
